In [1]:
import os
os.environ["USE_SYMENGINE"] = "1"

from sympy.core.backend import *
from delierium.matrix_order import Context, Mgrlex, Mgrevlex, Mlex
from delierium.JanetBasis import Janet_Basis, LHDP
from delierium.helpers import latexer
from IPython.display import Math

def Mlex(funcs, variables):  # pylint: disable=C0103
    '''Generates the "cotes" according to Riquier for the lex ordering
    INPUT : funcs: a tuple of functions (tuple for caching reasons)
            variables: a tuple of variables
            these are not used directly , just their lenght is interasting, but
            so the consumer doesn't has the burden of computing the length of
            list but the lists directly from context
    OUTPUT: a matrix which when multiplying an augmented vector (func + var)
            gives the vector in lex order

            same applies mutatis mutandis for Mgrlex and Mgrevlex

    >>> x,y,z = symbols("x y z")
    >>> f = Function("f")(x,y,z)
    >>> g = Function("g")(x,y,z)
    >>> h = Function("h")(x,y,z)
    >>> print(Mlex ((f,g), [x,y,z]))
    Matrix([[0, 0, 0, 2, 1], [1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0]])
    >>> x,y = symbols("x y")
    >>> w = Function("w")(x,y)
    >>> z = Function("z")(x,y)
    >>> print(Mlex((z,w), (x,y)))
    Matrix([[0, 0, 2, 1], [1, 0, 0, 0], [0, 1, 0, 0]])
    '''
    no_funcs = len(funcs)
    no_vars = len(variables)
    i = eye(no_vars)
    i = i.row_insert(0, Matrix(1, no_vars, [0]*no_vars))
    for j in range(no_funcs, 0, -1):
        i = i.row_join(Matrix([j] + [0]*no_vars))
    return i

In [2]:
x,y = symbols("x y")
w = Function("w")(x,y)
z = Function("z")(x,y)

In [3]:
Mlex((z,w), (y,x))

[0, 0, 2, 1]
[1, 0, 0, 0]
[0, 1, 0, 0]

In [4]:
Mlex((z,w), (x,y))

[0, 0, 2, 1]
[1, 0, 0, 0]
[0, 1, 0, 0]

In [31]:
ctx=Context((z, w), (y, x), Mlex)

In [32]:
diffs=[diff(w,y,y),
       diff(w,y,x),
       diff(w,x,x),
       diff(z,x,x),
       diff(z,y,y),
       diff(z,y,x),
       diff(z, x),
       diff(z,y),
       diff(w,y),
       diff(w,x),
       z,
       w]
diffs=[
       diff(w,y,x),
       diff(w,x,x),
       diff(w,y),
       z,
       ]
l=[LHDP(_, ctx) for _ in diffs]

In [33]:
print(l)

[D(w(x, y), x, y), D(w(x, y), x, x), D(w(x, y), y), z(x, y)]


In [34]:
l1=list(sorted(l, reverse = False))

In [35]:
from pprint import pprint
pprint(l1)

[D(w(x, y), x, x), D(w(x, y), y), D(w(x, y), x, y), z(x, y)]


In [36]:
l2=list(sorted(l, reverse = True))
pprint(l2)

[z(x, y), D(w(x, y), x, y), D(w(x, y), y), D(w(x, y), x, x)]
